In [ ]:
import os
import math
import time
import requests
import pandas as pd
from dotenv import load_dotenv

In [ ]:
"""
카카오 로컬 API + 등산로 기준 5km 반경 장소 수집 스크립트 (출발/도착, 3x3 그리드)

기능 요약
- mountain_round.txt(파일명, 출발_lat, 출발_lon, 도착_lat, 도착_lon) 읽기
- 각 등산로에 대해:
    - 출발 기준(base_type='start'), 도착 기준(base_type='end') 각각 수집
- 반경 5km 내 place 개수(카테고리별)를 먼저 확인
    - total_count <= 45 : 반경 5km 원으로만 수집
    - total_count > 45  : 10km x 10km 영역을 3x3 그리드(rect)로 나눠서 수집
- 수집 카테고리 : 음식점(FD6), 숙박(AD5), 카페(CE7)
- 각 기준 좌표마다 id 기준 중복 제거
- 거리(distance_m)는 항상 기준좌표(출발/도착) 기준 하버사인 거리
- distance_m <= 5000 인 장소만 남김
- 등산로별 결과 파일:
    - 폴더: ./result
    - 파일명: "{등산로명}_grid_3x3.csv"
    - 컬럼만 유지:
        mountain_name, gpx_file_name, base_type,
        place_id, place_name, category_group_name,
        lat, lng, distance_m, road_address_name, phone, url
- 전체 요약 파일:
    - ./result/summary_3x3.csv
    - 컬럼: mountain(gpx 파일명), category(카테고리명), count(고유 place 수)
- 통합 파일:
    - ./result/등하산_통합본.csv
    - 모든 개별 파일을 하나로 합친 파일
"""



# =====================
# 0. 설정값
# =====================

# .env 파일에서 API 키 로드
load_dotenv()
KAKAO_API_KEY = os.getenv('KAKAO_API_KEY')

# API 키 유효성 검사
api_key_valid = KAKAO_API_KEY and 'YOUR_API_KEY' not in KAKAO_API_KEY
print(f"API 키 설정 확인: {'✓' if api_key_valid else '✗'}")
if not api_key_valid:
    print("⚠️  .env 파일에서 KAKAO_API_KEY를 실제 API 키로 설정해주세요!")
    raise ValueError("KAKAO_API_KEY를 실제 카카오 REST API 키로 변경해주세요.")

MOUNTAIN_ROUND_FILE = "mountain_round.txt"
MOUNTAIN_ONEWAY_FILE = "mountain_oneway.txt"              # 각 줄: 코스명<TAB>lat<TAB>lon
TOUR_SPOT_FILE = "관광지_base.csv"       # 관광지 전체 데이터
OUTPUT_DIR = "result"

RADIUS_M = 5000        # 기준 반경 5km
GRID_N = 3             # 3x3 그리드
SLEEP_SEC = 0.2        # 요청 간 딜레이

HEADERS = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}

# 카테고리 그룹 코드 (카카오 공식)
CATEGORY_CODES = {
    "FD6": "음식점",
    "AD5": "숙박",
    "CE7": "카페",
}

TOUR_REQUIRED_COLS = [
    "poi_nm", "mcate_nm", "sido_nm", "sgg_nm",
    "bemd_nm", "ri_nm", "beonji",
    "rd_nm", "bld_num",
    "y", "x"
]

PAGE_SIZE = 15
MAX_PAGES = 5

# grid_scan을 적용할 카테고리
GRID_ENABLED_CATS = {"음식점", "카페", "숙박"}

# 샘플 등산로 개수 (None이면 전체)
MAX_MOUNTAINS = None   # 예: 5로 두면 5개만 테스트, 전체 돌릴 땐 None

BASE_URL = "https://dapi.kakao.com/v2/local/search/category.json"

# 요약 정보를 담을 전역 리스트
SUMMARY_ROWS = []   # 각 원소: DataFrame (mountain, category, count)
ALL_PLACES_ROWS = []  # 통합 파일용 전역 리스트



In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    """
    두 위경도 좌표 사이 거리(m) 계산 (하버사인 공식)
    """

    if lat1 is None or lon1 is None or lat2 is None or lon2 is None:
        return None

    R = 6371000  # 지구 반지름(m)
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c


def make_grid_rects(center_lat, center_lng, radius_m=5000, n=3):
    """
    중심 좌표를 기준으로 2*radius 크기의 정사각형을 만들고,
    이를 n x n 그리드로 나눈 각 셀의 rect(min_lng, min_lat, max_lng, max_lat) 리스트 반환
    (여기서는 n=3 → 3x3)
    """
    cell_size = (2 * radius_m) / n  # 한 셀 한 변 길이(m)

    # 위도/경도 1m 당 변화량(대략식)
    lat_per_m = 1 / 111320
    lon_per_m = 1 / (111320 * math.cos(math.radians(center_lat)))

    rects = []
    half = (n - 1) / 2  # n=3 → 1.0 → -1, 0, 1

    for i in range(n):      # y 방향 (위도)
        for j in range(n):  # x 방향 (경도)
            dx = (j - half) * cell_size
            dy = (i - half) * cell_size

            c_lat = center_lat + dy * lat_per_m
            c_lng = center_lng + dx * lon_per_m

            half_cell = cell_size / 2
            dlat = half_cell * lat_per_m
            dlng = half_cell * lon_per_m

            min_lat = c_lat - dlat
            max_lat = c_lat + dlat
            min_lng = c_lng - dlng
            max_lng = c_lng + dlng

            rects.append((min_lng, min_lat, max_lng, max_lat))

    return rects


# =====================
# 2. 카카오 API 호출 함수
# =====================

def kakao_category_circle_first_page(lat, lng, category_code, radius_m=5000):
    """
    반경 검색: 첫 페이지만 호출해서 meta와 docs 반환
    → total_count로 45개 초과 여부 판단에 사용
    """
    params = {
        "category_group_code": category_code,
        "x": lng,
        "y": lat,
        "radius": radius_m,
        "page": 1,
        "size": 15,  # 최대 15
        "sort": "distance",
    }
    resp = requests.get(BASE_URL, headers=HEADERS, params=params)
    resp.raise_for_status()
    data = resp.json()
    return data.get("meta", {}), data.get("documents", [])


def kakao_category_circle_all(lat, lng, category_code, radius_m=5000):
    """
    반경 검색: total_count <= 45인 경우, 가능한 모든 페이지(최대 3페이지) 수집
    """
    all_docs = []
    page = 1
    while True:
        params = {
            "category_group_code": category_code,
            "x": lng,
            "y": lat,
            "radius": radius_m,
            "page": page,
            "size": 15,
            "sort": "distance",
        }
        resp = requests.get(BASE_URL, headers=HEADERS, params=params)
        resp.raise_for_status()
        data = resp.json()
        docs = data.get("documents", [])
        meta = data.get("meta", {})
        all_docs.extend(docs)
        if meta.get("is_end") or not docs:
            break
        page += 1
        if page > 3:  # Kakao pageable_count 최대 45 → 3페이지
            break
        time.sleep(SLEEP_SEC)
    return all_docs


def kakao_category_rect_all(rect, category_code):
    """
    rect 영역 검색: min_lng, min_lat, max_lng, max_lat
    (각 rect당 최대 45개까지만 수집)
    """
    min_lng, min_lat, max_lng, max_lat = rect
    all_docs = []
    page = 1
    while True:
        params = {
            "category_group_code": category_code,
            "rect": f"{min_lng},{min_lat},{max_lng},{max_lat}",
            "page": page,
            "size": 15,
        }
        resp = requests.get(BASE_URL, headers=HEADERS, params=params)
        resp.raise_for_status()
        data = resp.json()
        docs = data.get("documents", [])
        meta = data.get("meta", {})
        all_docs.extend(docs)
        if meta.get("is_end") or not docs:
            break
        page += 1
        if page > 3:
            break
        time.sleep(SLEEP_SEC)
    return all_docs


# =====================
# 3. mountain_round.txt 읽기
# =====================

def load_trails(txt_path):
    """
    mountain_round.txt 읽어서 리스트 반환
    반환 형식: [{ 'file_name': ..., 'start_lat': float, 'start_lng': float, 'end_lat': float, 'end_lng': float }]
    """
    trails = []
    with open(txt_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    # 첫 줄은 헤더로 가정
    for line in lines[1:]:
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) < 5:
            continue
        file_name = parts[0]
        try:
            start_lat = float(parts[1])
            start_lng = float(parts[2])
            end_lat = float(parts[3])
            end_lng = float(parts[4])
        except ValueError:
            continue

        trails.append(
            {
                "file_name": file_name,
                "start_lat": start_lat,
                "start_lng": start_lng,
                "end_lat": end_lat,
                "end_lng": end_lng,
            }
        )
    return trails


# =====================
# 4. 기준좌표 단위 수집 로직
# =====================

def collect_places_for_base(mountain_name, file_name, base_type, base_lat, base_lng):
    """
    특정 기준좌표(출발/도착)에 대해 카테고리별 장소 정보 수집
    - base_type: 'start' 또는 'end'
    - 거리(distance_m)는 항상 base_lat/base_lng 기준 하버사인 거리
    - distance_m <= RADIUS_M 인 곳만 남김
    - 기준(base_type) 단위로 id 중복 제거
    """
    print(f"  - base={base_type:5s} | lat={base_lat:.5f}, lon={base_lng:.5f}")

    place_dict = {}  # key: place_id

    for cat_code, cat_name in CATEGORY_CODES.items():
        try:
            meta, _ = kakao_category_circle_first_page(
                base_lat, base_lng, cat_code, radius_m=RADIUS_M
            )
        except Exception as e:
            print(f"    * {cat_name:<4} : ERROR(circle meta) → {e}")
            continue

        total_count = meta.get("total_count", 0)
        mode = "CIRCLE" if total_count <= 45 else "GRID3x3"

        # 데이터 수집
        try:
            if mode == "CIRCLE":
                docs = kakao_category_circle_all(base_lat, base_lng, cat_code, radius_m=RADIUS_M)
            else:
                rects = make_grid_rects(base_lat, base_lng, radius_m=RADIUS_M, n=GRID_N)
                docs = []
                for rect in rects:
                    docs.extend(kakao_category_rect_all(rect, cat_code))
                    time.sleep(SLEEP_SEC)
        except Exception as e:
            print(f"    * {cat_name:<4} : ERROR(fetch {mode}) → {e}")
            continue

        raw_cnt = len(docs)
        before_len = len(place_dict)

        # 결과 처리 + dedupe + 거리 필터
        for d in docs:
            place_id = d.get("id")
            if not place_id:
                continue

            try:
                place_lat = float(d.get("y"))
                place_lng = float(d.get("x"))
            except (TypeError, ValueError):
                continue

            # 기준좌표 기준 거리
            dist_m = haversine(base_lat, base_lng, place_lat, place_lng)

            # 반경 5km 이내만
            if dist_m > RADIUS_M:
                continue

            if place_id in place_dict:
                continue

            place_dict[place_id] = {
                "mountain_name": mountain_name,
                "gpx_file_name": file_name,
                "base_type": base_type,          # start / end
                "place_id": place_id,
                "place_name": d.get("place_name"),
                "category_group_name": d.get("category_group_name"),
                "lat": place_lat,
                "lng": place_lng,
                "distance_m": dist_m,
                "road_address_name": d.get("road_address_name"),
                "phone": d.get("phone"),
                "url": d.get("place_url"),
            }

        kept_cnt = len(place_dict) - before_len
        print(
            f"    * {cat_name:<4} : mode={mode:<7} | total={total_count:3d} | "
            f"raw={raw_cnt:3d} → kept={kept_cnt:3d}"
        )

    print(f"    → base={base_type} unique places: {len(place_dict)}")
    return list(place_dict.values())


# =====================
# 5. 등산로별 처리
# =====================

def process_trail(trail):
    global SUMMARY_ROWS, ALL_PLACES_ROWS

    file_name = trail["file_name"]
    start_lat = trail["start_lat"]
    start_lng = trail["start_lng"]
    end_lat = trail["end_lat"]
    end_lng = trail["end_lng"]

    # 등산로명: 파일명에서 첫 '_' 앞 부분 사용
    base_name = os.path.basename(file_name)
    mountain_name = base_name.split("_")[0]

    print(f"\n=== Trail: {mountain_name} ({file_name}) ===")

    # 출발 기준 수집
    start_rows = collect_places_for_base(
        mountain_name, file_name, base_type="start",
        base_lat=start_lat, base_lng=start_lng
    )

    # 도착 기준 수집
    end_rows = collect_places_for_base(
        mountain_name, file_name, base_type="end",
        base_lat=end_lat, base_lng=end_lng
    )

    # start / end 결과 합치기
    all_rows = start_rows + end_rows
    total_places = len(all_rows)
    print(f"  → total rows(start+end): {total_places}")

    if not all_rows:
        print("  → no places found. skip saving.")
        return

    df = pd.DataFrame(all_rows)

    # 컬럼 순서를 네가 원하는 형태로 고정
    cols = [
        "mountain_name",
        "gpx_file_name",
        "base_type",
        "place_id",
        "place_name",
        "category_group_name",
        "lat",
        "lng",
        "distance_m",
        "road_address_name",
        "phone",
        "url",
    ]
    df = df[cols]

    # 등산로별 파일 저장 (즉시)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    safe_name = file_name.replace(".gpx", "")
    out_path = os.path.join(OUTPUT_DIR, f"{safe_name}_3x3.csv")

    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"  → saved: {out_path}")

    # 통합 파일용 데이터 추가
    ALL_PLACES_ROWS.append(df)

    # summary용 groupby (gpx 파일명 + 카테고리명)
    g = (
        df.groupby(["gpx_file_name", "category_group_name"])["place_id"]
          .nunique()
          .reset_index()
    )
    g.rename(columns={
        "gpx_file_name": "mountain",
        "category_group_name": "category",
        "place_id": "count",
    }, inplace=True)

    SUMMARY_ROWS.append(g[["mountain", "category", "count"]])



def main_round_trails():
    trails = load_trails(MOUNTAIN_ROUND_FILE)
    print(f"총 등산로 개수: {len(trails)}")

    for idx, trail in enumerate(trails, start=1):
        print(f"\n##### [{idx}/{len(trails)}] START #####")
        process_trail(trail)
        time.sleep(SLEEP_SEC)

    # 전체 summary 저장
    if SUMMARY_ROWS:
        summary_df = pd.concat(SUMMARY_ROWS, ignore_index=True)
        # 같은 mountain/category가 여러 번 있으면 합산
        summary_df = (
            summary_df.groupby(["mountain", "category"], as_index=False)["count"]
                     .sum()
        )
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        summary_path = os.path.join(OUTPUT_DIR, "summary_3x3.csv")
        summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
        print(f"\n=== Summary saved: {summary_path} ===")
    else:
        print("\n=== No data collected. Summary not created. ===")

    # 통합 파일 저장
    if ALL_PLACES_ROWS:
        all_places_df = pd.concat(ALL_PLACES_ROWS, ignore_index=True)
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        all_places_path = os.path.join(OUTPUT_DIR, "등하산_통합본.csv")
        all_places_df.to_csv(all_places_path, index=False, encoding="utf-8-sig")
        print(f"=== All places saved: {all_places_path} (total rows: {len(all_places_df)}) ===")
    else:
        print("=== No places data collected. All places file not created. ===")


def kakao_search_radius_base(lat, lon, category_code, radius):
    """
    반경 radius 내에서 카카오 카테고리 검색 (페이지네이션 포함)
    반환: (documents list, meta dict)
    """
    results = []
    first_meta = None

    for page in range(1, MAX_PAGES + 1):
        params = {
            "category_group_code": category_code,
            "x": lon,
            "y": lat,
            "radius": radius,
            "page": page,
            "size": PAGE_SIZE,
        }

        try:
            resp = requests.get(BASE_URL, headers=HEADERS, params=params, timeout=10)
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            print(f"    [HTTP ERROR] {e}")
            break

        if first_meta is None:
            first_meta = data.get("meta", {})

        docs = data.get("documents", [])
        if not docs:
            break

        results.extend(docs)

        meta = data.get("meta", {})
        if meta.get("is_end", True) or len(docs) < PAGE_SIZE:
            break

        time.sleep(SLEEP_SEC)

    return results, (first_meta or {})


# ==========================
# 3. grid_scan (3x3, small_radius=2000 고정)
# ==========================
def grid_scan(lat_center, lon_center, category_code, big_radius=RADIUS_M, small_radius=2000):
    """
    big_radius 내를 small_radius 원 3x3으로 나눠서 검색 (grid_range=1 -> 3x3)
    small_radius는 2000으로 고정(요청사항)
    """
    results = []

    # 위도/경도 변환 근사: 1km ≈ 0.009도
    delta = (small_radius / 1000) * 0.009
    grid_range = 1  # -1,0,1 => 3x3

    print(f"  [grid_scan] big_radius={big_radius}m, small_radius={small_radius}m, grid=3x3")

    for dx in range(-grid_range, grid_range + 1):
        for dy in range(-grid_range, grid_range + 1):
            lat = lat_center + dy * delta
            lon = lon_center + dx * delta

            docs, _ = kakao_search_radius_base(lat, lon, category_code, small_radius)
            print(f"    grid({dx},{dy}) → {len(docs)}개")
            results.extend(docs)
            time.sleep(SLEEP_SEC)

    # id 기준 중복 제거 (가장 신뢰할 수 있는 키)
    dedup = {}
    for d in results:
        pid = d.get("id")
        if pid and pid not in dedup:
            dedup[pid] = d

    merged_docs = list(dedup.values())
    print(f"  [grid_scan] 통합 후 중복 제거: {len(merged_docs)}개")

    return merged_docs


# ==========================
# 4. docs → DataFrame 표준화
# ==========================
def normalize_docs(docs, mountain_name, center_type, category_label, center_lat, center_lon):
    rows = []
    for d in docs:
        try:
            y = float(d.get("y")) if d.get("y") is not None else None
            x = float(d.get("x")) if d.get("x") is not None else None
        except ValueError:
            y, x = None, None

        dist = haversine(center_lat, center_lon, y, x) if (y is not None and x is not None) else None

        rows.append({
            "mountain": mountain_name,                     # 파일명 그대로
            "center_type": center_type,                    # 빈 문자열 또는 "center"
            "category": category_label,                    # 음식점 / 카페 / 숙박 / 관광명소
            "id": d.get("id"),
            "place_name": d.get("place_name", ""),
            "address_name": d.get("address_name", ""),
            "road_address_name": d.get("road_address_name", ""),
            "tel": d.get("phone", ""),
            "x": x,                                        # lon
            "y": y,                                        # lat
            "distance_m": dist,                            # 중심 좌표로부터 haversine 거리 (m)
            "kakao_url": d.get("place_url", ""),
        })
    return pd.DataFrame(rows)


# ==========================
# 5. 한 좌표(코스) 당 카테고리별 수집
# ==========================
def process_one_center(mountain_name, lat0, lon0):
    """
    단일 중심 좌표(위도 lat0, 경도 lon0)에 대해
    - 음식점 / 카페 / 숙박 카테고리 수집
    - RADIUS_M 기본 검색 후 pageable_count >=45이면 grid_scan 수행
    """
    dfs = []

    for cat_code, cat_name in CATEGORY_CODES.items():
        # 반경 적용
        radius = RADIUS_M

        print(f" ▶ [{mountain_name}] {cat_name} 기본 검색 (반경 {radius}m)")

        try:
            base_docs, meta = kakao_search_radius_base(lat0, lon0, cat_code, radius)
        except Exception as e:
            print(f"   [요청 실패] {cat_name}: {e}")
            continue

        total_count = meta.get("total_count", 0)
        pageable_count = meta.get("pageable_count", 0)
        print(f"    - base_docs: {len(base_docs)}개 / total_count: {total_count} / pageable_count: {pageable_count}")

        final_docs = base_docs

        # grid_scan 조건: 카테고리 적용 대상이고 5km 검색이며 pageable_count >= 45
        if (cat_name in GRID_ENABLED_CATS) and (radius == RADIUS_M) and (pageable_count >= 45):
            print(f"    → pageable_count >= 45 이므로 grid_scan 수행")
            grid_docs = grid_scan(lat0, lon0, cat_code, big_radius=RADIUS_M, small_radius=2000)

            # base_docs + grid_docs 합치고 id 기준으로 중복 제거
            all_docs = base_docs + grid_docs
            dedup = {}
            for d in all_docs:
                pid = d.get("id")
                if pid and pid not in dedup:
                    dedup[pid] = d
            final_docs = list(dedup.values())
            print(f"    → base + grid 통합 후: {len(final_docs)}개")

        df_cat = normalize_docs(final_docs, mountain_name, "center", cat_name, lat0, lon0)

        # 내부 중복 제거(안전장치)
        if not df_cat.empty:
            if "id" in df_cat.columns and not df_cat["id"].isnull().all():
                df_cat = df_cat.drop_duplicates(subset=["id"], keep="first")
            else:
                df_cat = df_cat.drop_duplicates(subset=["place_name", "address_name"], keep="first")

        dfs.append(df_cat)
        time.sleep(0.15)

    # 모든 카테고리 concat
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame([])


# ==========================
# 6. main: mountain_oneway.txt 읽기(단일 좌표) + 저장
# ==========================
def main_oneway_grid():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # 1) mountain_oneway.txt 읽기: 각 줄은 "파일명<TAB>lat<TAB>lon"
    centers = []
    with open(MOUNTAIN_ONEWAY_FILE, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if not parts or len(parts) < 3:
                print(f"[WARN] 라인 형식 오류(스킵): {line.strip()}")
                continue
            name = parts[0].strip()
            try:
                lat = float(parts[1])
                lon = float(parts[2])
            except:
                print(f"[WARN] 위경도 변환 실패(스킵): {line.strip()}")
                continue
            centers.append((name, lat, lon))

    if MAX_MOUNTAINS is not None:
        centers = centers[:MAX_MOUNTAINS]

    print(f"[INFO] 처리할 중심점 개수: {len(centers)}")

    all_places_list = []
    all_summary_list = []

    # 2) 각 중심(코스) 처리
    for idx, (fname, lat0, lon0) in enumerate(centers, start=1):
        print(f"\n=== [{idx}/{len(centers)}] 처리: {fname} ({lat0},{lon0}) ===")

        df_m = process_one_center(fname, lat0, lon0)

        if df_m.empty:
            print("  → 수집된 데이터가 없습니다.")
            continue

        # 파일명에서 .gpx 등 확장자 제거
        mountain_base = fname.split(".")[0]

        # 3-1) 이 코스의 전체 장소 데이터 저장 (단일 CSV)
        places_path = os.path.join(OUTPUT_DIR, f"{mountain_base}_places_grid.csv")
        df_m.to_csv(places_path, index=False, encoding="utf-8-sig")
        print(f"  [저장] places → {places_path} (행 {len(df_m)}개)")

        # 3-2) 이 코스의 요약 데이터 (category별 고유 id 수)
        summary = (
            df_m.groupby("category")["id"]
                .nunique()
                .reset_index()
                .rename(columns={"id": "count"})
        )
        summary.insert(0, "mountain", fname)

        summary_path = os.path.join(OUTPUT_DIR, f"{mountain_base}_summary_grid.csv")
        summary.to_csv(summary_path, index=False, encoding="utf-8-sig")
        print(f"  [저장] summary → {summary_path} (행 {len(summary)}개)")

        # 전체 합본용 리스트에 추가
        all_places_list.append(df_m)
        all_summary_list.append(summary)

        time.sleep(0.5)

    # 4) 전체 합본 파일 저장
    if all_places_list:
        df_all_places = pd.concat(all_places_list, ignore_index=True)
        all_places_path = os.path.join(OUTPUT_DIR, "편도_통합본.csv")
        df_all_places.to_csv(all_places_path, index=False, encoding="utf-8-sig")
        print(f"\n[저장] 전체 places 합본 → {all_places_path} (행 {len(df_all_places)}개)")
    else:
        print("\n[알림] 전체 places 데이터 없음")

    if all_summary_list:
        df_all_summary = pd.concat(all_summary_list, ignore_index=True)
        all_summary_path = os.path.join(OUTPUT_DIR, "summary_all_grid.csv")
        df_all_summary.to_csv(all_summary_path, index=False, encoding="utf-8-sig")
        print(f"[저장] 전체 summary 합본 → {all_summary_path} (행 {len(df_all_summary)}개)")
    else:
        print("[알림] 전체 summary 데이터 없음")



def _postprocess_tour_df(df_out: pd.DataFrame, *, is_round: bool) -> pd.DataFrame:
    """공통 후처리(주소, rename, drop, 정렬)"""
    if df_out.empty:
        return df_out

    df_out["address"] = (
        df_out["sido_nm"].fillna("") + " " +
        df_out["sgg_nm"].fillna("") + " " +
        df_out["bemd_nm"].fillna("") + " " +
        df_out["rd_nm"].fillna("") + " " +
        df_out["bld_num"].fillna("")
    ).str.strip()

    df_out.rename(columns={
        "poi_nm": "place_name",
        "mcate_nm": "category",
        "y": "lat",
        "x": "lng",
    }, inplace=True)

    df_out.drop(
        columns=["sido_nm", "sgg_nm", "bemd_nm", "rd_nm", "bld_num", "ri_nm", "beonji"],
        errors="ignore",
        inplace=True
    )

    cols = df_out.columns.tolist()
    if "mountain_point" in cols:
        cols.remove("mountain_point")
        cols.insert(0, "mountain_point")
        df_out = df_out[cols]

    if is_round and "base_type" in df_out.columns:
        df_out.sort_values(by=["base_type", "distance_m"], inplace=True)
    else:
        df_out.sort_values(by="distance_m", inplace=True)

    return df_out


def run_tourist_pipeline(*, mode: str):
    """
    mode:
      - "oneway" : MOUNTAIN_ONEWAY_FILE (탭 3개)
      - "round"  : MOUNTAIN_ROUND_FILE  (탭 5개)
    """
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # 1) 산 목록 로드
    mountains = []
    if mode == "oneway":
        with open(MOUNTAIN_ONEWAY_FILE, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split("\t")
                if len(parts) < 3:
                    continue
                try:
                    name = parts[0]
                    lat = float(parts[1]); lon = float(parts[2])
                    mountains.append((name, lat, lon))
                except ValueError:
                    continue
    elif mode == "round":
        with open(MOUNTAIN_ROUND_FILE, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split("\t")
                if len(parts) < 5:
                    continue
                try:
                    name = parts[0]
                    s_lat = float(parts[1]); s_lon = float(parts[2])
                    e_lat = float(parts[3]); e_lon = float(parts[4])
                    mountains.append((name, s_lat, s_lon, e_lat, e_lon))
                except ValueError:
                    continue
    else:
        raise ValueError("mode는 'oneway' 또는 'round'여야 합니다.")

    print(f"산 데이터 {len(mountains)}개 로드됨 (mode={mode})")

    # 2) 관광지 전체 로드
    df_tour = pd.read_csv(TOUR_SPOT_FILE)
    print(f"관광지 전체 {len(df_tour)}개 로드됨")

    summary = []
    all_tourist_rows = []

    # 3) 산별 처리
    if mode == "oneway":
        for name, lat, lon in mountains:
            print(f"\n▶ {name} 처리 중...")
            rows = []

            for _, row in df_tour.iterrows():
                try:
                    tlat = float(row["y"]); tlon = float(row["x"])
                except:
                    continue

                dist = haversine(lat, lon, tlat, tlon)
                if dist <= RADIUS_M:
                    extracted = {c: row[c] for c in TOUR_REQUIRED_COLS if c in row}
                    extracted["distance_m"] = dist
                    extracted["mountain_point"] = name
                    rows.append(extracted)

            df_out = pd.DataFrame(rows)
            df_out = _postprocess_tour_df(df_out, is_round=False)

            path = os.path.join(OUTPUT_DIR, f"{name}_관광지.csv")
            df_out.to_csv(path, index=False, encoding="utf-8-sig")
            print(f"   → 저장 완료: {path} ({len(df_out)}개)")

            summary.append({"mountain_point": name, "count": len(df_out)})
            if not df_out.empty:
                all_tourist_rows.append(df_out)

        summary_file = "편도_summary.csv"
        all_file = "편도_all_tourist.csv"

    else:  # round
        for name, s_lat, s_lon, e_lat, e_lon in mountains:
            print(f"\n▶ {name} 처리 중...")
            rows = []

            for base_type, lat, lon in [("start", s_lat, s_lon), ("end", e_lat, e_lon)]:
                for _, row in df_tour.iterrows():
                    try:
                        tlat = float(row["y"]); tlon = float(row["x"])
                    except:
                        continue

                    dist = haversine(lat, lon, tlat, tlon)
                    if dist <= RADIUS_M:
                        extracted = {c: row[c] for c in TOUR_REQUIRED_COLS if c in row}
                        extracted["distance_m"] = dist
                        extracted["base_type"] = base_type
                        extracted["mountain_point"] = name
                        rows.append(extracted)

            df_out = pd.DataFrame(rows)
            df_out = _postprocess_tour_df(df_out, is_round=True)

            path = os.path.join(OUTPUT_DIR, f"{name}_관광지.csv")
            df_out.to_csv(path, index=False, encoding="utf-8-sig")
            print(f"   → 저장 완료: {path} ({len(df_out)}개)")

            summary.append({"mountain_point": name, "count": len(df_out)})
            if not df_out.empty:
                all_tourist_rows.append(df_out)

        summary_file = "왕복_summary.csv"
        all_file = "왕복_all_tourist.csv"

    # 4) 요약/합본 저장
    pd.DataFrame(summary).to_csv(
        os.path.join(OUTPUT_DIR, summary_file),
        index=False,
        encoding="utf-8-sig"
    )
    print(f"\n=== 요약 저장 완료: {summary_file} ({len(summary)}개) ===")

    if all_tourist_rows:
        pd.concat(all_tourist_rows, ignore_index=True).to_csv(
            os.path.join(OUTPUT_DIR, all_file),
            index=False,
            encoding="utf-8-sig"
        )
        print(f"=== 전체 관광지 합본 저장 완료: {all_file} ===")

    print("\n=== 모든 작업 완료 ===")

def main_round_tourist():
    run_tourist_pipeline(mode="round")

def main_oneway_tourist():
    run_tourist_pipeline(mode="oneway")


## 1. 등산로 반경 5km 관광인프라 poi 수집

---

**주요 기능:**

- 카카오 로컬 API를 사용해 각 지점(출발지/도착지) 5km 반경 내 음식점/카페/숙박업소 검색

- 업소명, 좌표, 거리, 주소, 연락처 등 상세 정보 수집
  
- 각 기준좌표에서 반경 5km 검색으로 total_count를 확인해 수집 방식 결정

  - `total_count ≤ 45` → 반경 5km 원형 검색 수집

  - `total_count > 45` → 10km×10km 영역을 3×3 그리드(rect) 로 나눠서 수집(셀당 최대 45개)

In [ ]:
if __name__ == "__main__":
    main_round_trails()

### 편도 등산로

In [ ]:
if __name__ == "__main__":
    main_oneway_grid()

## 2. 관광명소 공공데이터기반 : 등산로 반경 5km 내 관광명소 매칭

---

**주요 기능:**

- 각 지점(출발지/도착지)을 기준으로 관광명소와의 거리(하버사인) 계산

- 반경 5km 이내 관광명소만 필터링해 저장


### 등하산(왕복) 등산로

In [ ]:
if __name__ == "__main__":
    main_round_tourist()

### 편도 등산로

In [ ]:
if __name__ == "__main__":
    main_oneway_tourist()